In [ ]:
import os
import pickle
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

tar_path = Path(os.environ["GAITLU_RAW_ROOT"])
output_path = Path(os.environ.get("GAITLU_OUTPUT", "gaitlu-preview.png"))

n_sequences = 4
n_frames = 8


def decode_sequence(value):
    if isinstance(value, dict):
        value = value.get("silhouettes", value.get("frames", value.get("data")))

    if isinstance(value, (list, tuple)) and len(value) == 1:
        value = value[0]

    frames = np.asarray(value)

    if frames.ndim == 4 and frames.shape[1] == 1:
        frames = frames[:, 0]
    elif frames.ndim == 4 and frames.shape[-1] == 1:
        frames = frames[..., 0]

    if frames.ndim != 3:
        raise ValueError(f"Expected [T, H, W], got {frames.shape}")

    if frames.max() <= 1:
        return frames >= 0.5
    return frames >= 128


with tarfile.open(tar_path, "r:*") as archive:
    members = [
        member for member in archive.getmembers()
        if member.isfile() and member.name.endswith(".pkl")
    ]

    if not members:
        raise RuntimeError(
            "This shard contains no .pkl files. It may be a bit-packed "
            "prepared shard requiring an inventory CSV."
        )

    selected = members[:n_sequences]
    videos = []

    for member in selected:
        with archive.extractfile(member) as handle:
            value = pickle.load(handle)  # use only with trusted dataset files
        videos.append((member.name, decode_sequence(value)))


fig, axes = plt.subplots(
    len(videos),
    n_frames,
    figsize=(2 * n_frames, 2.5 * len(videos)),
    squeeze=False,
)

for row_index, (name, video) in enumerate(videos):
    indices = np.linspace(0, len(video) - 1, n_frames).astype(int)

    for column_index, frame_index in enumerate(indices):
        ax = axes[row_index, column_index]
        ax.imshow(
            video[frame_index],
            cmap="gray",
            vmin=0,
            vmax=1,
            interpolation="nearest",
        )
        ax.axis("off")

        if row_index == 0:
            ax.set_title(f"frame {frame_index}")

    axes[row_index, 0].set_ylabel(
        Path(name).parent.name,
        rotation=0,
        labelpad=35,
        va="center",
    )

fig.suptitle(f"GaitLU preview: {tar_path.name}")
plt.tight_layout()

output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, dpi=150, bbox_inches="tight")
print(f"Saved image to {output_path}")
plt.show()